# Retrieval Evaluation (BM25 vs TF-IDF)

Load the BM25 index and the TF-IDF index -> retrieve for each query in `qa_retrieval.jsonl` -> compute Recall@10/20/50 / MRR@20 / nDCG@20 / MAP@20 for both retrievers.


In [ ]:
import os, sys
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if os.path.exists(os.path.join(p, "src", "rag")):
        sys.path.insert(0, p)
        break

from src.rag import config
from src.rag.data import loader
from src.rag.retrieval import bm25, tfidf
from src.rag.eval import metrics


In [ ]:
bm25_idx = bm25.load_index(config.PKG_ROOT / "index" / "bm25.pkl")
tfidf_idx = tfidf.load_index(config.PKG_ROOT / "index" / "tfidf.pkl")

qa = loader.load_qa_retrieval()
print("num eval queries:", len(qa))


In [ ]:
all_gold = [q["gold_pids"] for q in qa]

rankings = {"BM25": [], "TF-IDF": []}
for q in qa:
    bm = bm25_idx.search(q["question"], k=config.TOP_K_EVAL)
    rankings["BM25"].append([pid for pid, _ in bm])

    tf = tfidf_idx.search(q["question"], k=config.TOP_K_EVAL)
    rankings["TF-IDF"].append([pid for pid, _ in tf])


In [ ]:
metrics_order = ["recall@10", "recall@20", "recall@50", "mrr@20", "ndcg@20", "map@20"]

res = {name: metrics.evaluate(all_gold, ranked, k_values=(10, 20, 50), rank_k=20)
       for name, ranked in rankings.items()}

for m in metrics_order:
    print(f"{m:12s} " + "  ".join(f"{res[n][m]:.4f}" for n in res))

os.makedirs(config.RESULTS_DIR, exist_ok=True)
out = config.RESULTS_DIR / "retrieval_result.md"
with open(out, "w", encoding="utf-8") as f:
    f.write("# Retrieval ablation (BM25 / TF-IDF)\n\n")
    f.write("| metric | " + " | ".join(res.keys()) + " |\n")
    f.write("|---|" + "---|" * len(res) + "\n")
    for m in metrics_order:
        f.write(f"| {m} | " + " | ".join(f"{res[n][m]:.4f}" for n in res) + " |\n")
print("wrote:", out)
